# Topic analysis 2
## Expressions and evidence for a chosen partition

This notebook helps you identify recurrent themes and decide cluster names yourself.
It starts with an existing partition, or one you specify through clustering parameters.
It then reports **what expressions distinguish each cluster, how many papers support
them, and where they occur in the source text**. No proposed topic names are loaded or generated.

**Reading order:** choose a partition → compare clusters → review two example papers per cluster → inspect one cluster →
inspect an expression in its papers → optionally export the evidence.

The corpus is the same **292 canonical works** used in the saved thesis experiments.
This differs from the earlier 294-PDF selection because duplicate versions are merged
at work level. The notebook verifies the saved input identities before using a partition.
It uses existing text and vectors; it does not extract PDFs, load an embedding model,
call an API, or run the full cluster-selection experiment.

## 1 Load the corpus and list saved partitions

Use the project Python (batill) kernel from .venv/bin/python. The first load checks the
prepared text and vectors and may take a little time. The table lists exact saved
partitions; choosing one preserves its membership and displayed cluster IDs.

In [33]:
from pathlib import Path
from html import escape
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML

candidates = [Path.cwd(), Path.cwd() / 'batill', *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src/batill').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the batill project.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from batill.topic_explorer import (
    load_context, partition_catalog, choose_partition, analyse_partition,
    ranked_terms, evidence_table, supporting_papers, export_analysis,
)

context = load_context(ROOT)
print(f"Loaded {len(context['papers'])} works and their existing vectors.")
display(partition_catalog(context).style.hide(axis='index').format(precision=3, na_rep='—'))

Loaded 292 works and their existing vectors.


solution,method,clusters,assigned,unassigned,cosine_silhouette,seed_ARI,resolution
kmeans_k2,K-means,2,292,0,0.169,0.959,—
kmeans_k3,K-means,3,292,0,0.207,0.944,—
kmeans_k4,K-means,4,292,0,0.189,0.857,—
kmeans_k5,K-means,5,292,0,0.178,0.865,—
kmeans_k6,K-means,6,292,0,0.158,0.689,—
kmeans_k7,K-means,7,292,0,0.167,0.641,—
kmeans_k8,K-means,8,292,0,0.153,0.600,—
kmeans_k9,K-means,9,292,0,0.116,0.532,—
kmeans_k10,K-means,10,292,0,0.089,0.485,—
kmeans_k11,K-means,11,292,0,0.115,0.550,—


## 2 Choose the partition

Change **SOURCE** and the appropriate settings below, then run this cell and the
cells below it. The default is simply a starting example, not a new count recommendation.

| Source | Settings used | Behaviour |
|---|---|---|
| `saved` | `SAVED_PARTITION` | Loads the exact previously computed partition. All fit settings below are ignored. |
| `fit` with `kmeans` | `K`, `SEED`, `N_INIT` | Fits one K-means setting on the existing full-dimensional vectors. |
| `fit` with `leiden` | `RESOLUTION`, `NEIGHBOURS`, `SEED` | Fits text Leiden on a positive cosine k-nearest-neighbour graph. Resolution determines the resulting count; `K` is ignored. |

Saved examples: `kmeans_k4`, `leiden_k5`, `citation_r1`, `hierarchy_15`.
A fresh single-seed fit may differ from a saved result selected across several seeds.
Citation clustering is available here through its saved baseline, not through a new fit.

In [34]:
SOURCE = 'saved'                 # 'saved' or 'fit'
SAVED_PARTITION = 'leiden_k5'     # Copy a solution name from the table above.

METHOD = 'leiden'                # Fit mode only: 'kmeans' or 'leiden'.
K = 3
RESOLUTION = 0.95
NEIGHBOURS = 15
SEED = 42
N_INIT = 10

labels, partition_settings = choose_partition(
    context, source=SOURCE, saved=SAVED_PARTITION, method=METHOD,
    k=K, resolution=RESOLUTION, neighbours=NEIGHBOURS, seed=SEED, n_init=N_INIT,
)
print(json.dumps({k: v for k, v in partition_settings.items()
                  if k != 'original_settings'}, indent=2))
print(f"Assigned papers: {(labels > 0).sum()} | Unassigned: {(labels == 0).sum()}")
print(f"Clusters: {len(set(labels) - {0})}")

{
  "source": "saved",
  "saved": "leiden_k5",
  "method": "Text Leiden",
  "clusters": 5,
  "seed": 44,
  "resolution": 0.95,
  "selection": "highest seed ARI among exact-count candidates"
}
Assigned papers: 292 | Unassigned: 0
Clusters: 5


## 3 Compute the expression evidence

Text comes from the retained PDF source blocks across all sections. Repeated model
instructions and title prefixes are removed. Lowercasing and stopword removal are
applied; numerical tokens and formatting terms are excluded. Adjacent-word expressions
do not bridge punctuation, stopwords or source-block boundaries. Singular/plural forms
and synonyms remain separate.

Two rankings are computed on the same vocabulary:

- **TF-IDF contrast:** mean normalized per-paper TF-IDF inside the cluster minus the
  mean outside. A positive score indicates greater average TF-IDF weight inside.
- **Class-based TF-IDF:** relative pooled frequency within the cluster multiplied by
  `log(1 + mean cluster vocabulary length / corpus term count)`.

Vocabulary terms must occur in at least **3 papers**, and no more than **90%** of the
analysed papers. Ranked terms additionally require a positive score and support in at
least **10% of cluster papers, with a minimum of 2**. A singleton therefore cannot
produce a supported expression under this rule. No terms are removed manually to
make a cluster look more coherent.

For citation partitions, unassigned isolates are excluded from both the vocabulary
and the inside/outside denominators. The table below makes the analysed population explicit.

In [35]:
analysis = analyse_partition(context, labels, min_df=3, max_df=0.9)
print(f"Analysed papers: {len(analysis['papers'])}; vocabulary: {len(analysis['vocabulary']['terms']):,} terms")
display(analysis['summary'][['cluster', 'papers', 'minimum_support']].rename(columns={
    'cluster': 'Cluster', 'papers': 'Papers', 'minimum_support': 'Minimum supporting papers',
}).style.hide(axis='index'))
if len(analysis['unassigned']):
    display(Markdown('**Unassigned papers excluded from topic contrast:**'))
    display(analysis['unassigned'][['paper_id', 'title']].style.hide(axis='index'))

Analysed papers: 292; vocabulary: 44,195 terms


Cluster,Papers,Minimum supporting papers
1,73,8
2,62,7
3,31,4
4,97,10
5,29,3


### Search a word or expression across clusters

Enter a term below and rerun only this cell to compare its frequency across the
chosen partition. **Papers containing term** counts each paper once; **total
occurrences** includes every repetition. The percentage uses the size of each
cluster; occurrences per paper includes papers with zero matches.

Search is case-insensitive and matches whole words in sequence. Hyphens are treated
as spaces; singular/plural forms are separate. Phrases cannot cross sentence or
source-block boundaries. Unlike ranked vocabulary, manual search also accepts rare
terms, stopwords, numbers and phrases longer than two words, without support filters.
Unassigned papers excluded in part 3 remain excluded here. Raw occurrence counts
are sensitive to document length and repeated table text.

In [36]:
# Reload the helper so this new cell also works in an already-running kernel.
from importlib import reload
import batill.topic_explorer as topic_explorer
reload(topic_explorer)

SEARCH_TERM = 'private equity'   # Replace with any word or contiguous expression.
search_results = topic_explorer.search_expression(analysis, SEARCH_TERM)
print('Search:', search_results.attrs['normalized_query'])
display(search_results.style.hide(axis='index').format({
    'Papers containing term (%)': '{:.1f}', 'Occurrences per paper': '{:.2f}',
}))
print(f"Across all analysed clusters: {search_results['Papers containing term'].sum()} / "
      f"{search_results['Papers in cluster'].sum()} papers; "
      f"{search_results['Total occurrences'].sum()} occurrences.")

Search: private equity


Cluster,Papers in cluster,Papers containing term,Papers containing term (%),Total occurrences,Occurrences per paper
1,73,73,100.0,4462,61.12
2,62,44,71.0,1875,30.24
3,31,29,93.5,928,29.94
4,97,97,100.0,4393,45.29
5,29,29,100.0,1334,46.00


Across all analysed clusters: 272 / 292 papers; 12992 occurrences.


## 4 Compare the clusters without naming them

Each cluster receives its own small table. Start with two-word expressions, then
optionally switch `TERM_KIND` to `word`. Rankings are computed separately for these
display types without changing their scores. Change `RANKING` to compare methods.

| Column | Meaning |
|---|---|
| Score | Value under the selected ranking; scores from different ranking methods are not on the same scale. |
| Inside / outside papers | Number of papers containing the term at least once, divided by the corresponding population. |
| Inside / outside % | Paper-level prevalence, regardless of repetition within a paper. |
| Gap (pp) | Inside prevalence minus outside prevalence, in percentage points. |

A contrast score and a prevalence gap measure different things: TF-IDF also considers
repetition and document normalization. A positive contrast need not imply a positive
prevalence gap. These are **descriptive statistics, not significance tests**.

In [37]:
RANKING = 'tfidf_contrast'       # Or 'c_tf_idf'.
TERM_KIND = 'expression'        # Or 'word'.
TOP_N = 15

def show_terms(rows):
    if rows.empty:
        print('No terms meet the support and positive-score thresholds.')
        return
    display(evidence_table(rows).style.hide(axis='index').format({
        'Score': '{:.5f}', 'Inside %': '{:.1f}', 'Outside %': '{:.1f}', 'Gap (pp)': '{:+.1f}',
    }))

for row in analysis['summary'].itertuples():
    display(Markdown(f'### Cluster {row.cluster} — {row.papers} papers'))
    show_terms(ranked_terms(analysis, row.cluster, RANKING, TERM_KIND, TOP_N))

### Cluster 1 — 73 papers

Term,Score,Inside papers,Inside total,Outside papers,Outside total,Inside %,Outside %,Gap (pp)
pe backed,0.01379,33,73,25,219,45.2,11.4,+33.8
pe investors,0.01163,37,73,37,219,50.7,16.9,+33.8
employment growth,0.01143,24,73,7,219,32.9,3.2,+29.7
pe firms,0.01108,47,73,72,219,64.4,32.9,+31.5
pe ownership,0.00943,29,73,24,219,39.7,11.0,+28.8
backed firms,0.00931,30,73,28,219,41.1,12.8,+28.3
non pe,0.00915,33,73,27,219,45.2,12.3,+32.9
firm level,0.00902,43,73,45,219,58.9,20.5,+38.4
pe firm,0.00881,39,73,49,219,53.4,22.4,+31.1
control firms,0.00846,24,73,13,219,32.9,5.9,+26.9


### Cluster 2 — 62 papers

Term,Score,Inside papers,Inside total,Outside papers,Outside total,Inside %,Outside %,Gap (pp)
leveraged buyouts,0.01094,52,62,78,230,83.9,33.9,+50.0
industry adjusted,0.01063,22,62,17,230,35.5,7.4,+28.1
post buyout,0.01058,24,62,31,230,38.7,13.5,+25.2
capital expenditures,0.01056,22,62,30,230,35.5,13.0,+22.4
operating performance,0.01054,36,62,46,230,58.1,20.0,+38.1
book value,0.01053,34,62,37,230,54.8,16.1,+38.8
post lbo,0.01025,17,62,9,230,27.4,3.9,+23.5
operating cash,0.00990,20,62,7,230,32.3,3.0,+29.2
total debt,0.00984,27,62,20,230,43.5,8.7,+34.9
operating income,0.00984,24,62,22,230,38.7,9.6,+29.1


### Cluster 3 — 31 papers

Term,Score,Inside papers,Inside total,Outside papers,Outside total,Inside %,Outside %,Gap (pp)
going public,0.01410,15,31,26,261,48.4,10.0,+38.4
going private,0.01382,13,31,51,261,41.9,19.5,+22.4
private firms,0.01326,22,31,76,261,71.0,29.1,+41.8
firms going,0.01201,11,31,9,261,35.5,3.4,+32.0
public firms,0.01188,23,31,71,261,74.2,27.2,+47.0
private firm,0.01124,16,31,33,261,51.6,12.6,+39.0
compliance costs,0.01065,9,31,2,261,29.0,0.8,+28.3
sarbanes oxley,0.01036,11,31,6,261,35.5,2.3,+33.2
private decision,0.01007,6,31,2,261,19.4,0.8,+18.6
oxley act,0.00967,10,31,3,261,32.3,1.1,+31.1


### Cluster 4 — 97 papers

Term,Score,Inside papers,Inside total,Outside papers,Outside total,Inside %,Outside %,Gap (pp)
vintage year,0.01855,72,97,13,195,74.2,6.7,+67.6
buyout funds,0.01799,77,97,28,195,79.4,14.4,+65.0
vc funds,0.01781,56,97,10,195,57.7,5.1,+52.6
fund size,0.01717,81,97,27,195,83.5,13.8,+69.7
fund performance,0.01591,70,97,19,195,72.2,9.7,+62.4
committed capital,0.01381,65,97,12,195,67.0,6.2,+60.9
vintage years,0.01262,54,97,5,195,55.7,2.6,+53.1
capital funds,0.01211,61,97,18,195,62.9,9.2,+53.7
fund level,0.01189,67,97,30,195,69.1,15.4,+53.7
fund type,0.01100,44,97,1,195,45.4,0.5,+44.8


### Cluster 5 — 29 papers

Term,Score,Inside papers,Inside total,Outside papers,Outside total,Inside %,Outside %,Gap (pp)
health care,0.04092,26,29,23,263,89.7,8.7,+80.9
nursing homes,0.03184,18,29,6,263,62.1,2.3,+59.8
physician practices,0.02916,16,29,0,263,55.2,0.0,+55.2
nursing home,0.02885,14,29,3,263,48.3,1.1,+47.1
equity acquisition,0.02285,16,29,8,263,55.2,3.0,+52.1
care delivery,0.02100,17,29,0,263,58.6,0.0,+58.6
pe owned,0.01846,15,29,22,263,51.7,8.4,+43.4
acquired hospitals,0.01762,8,29,0,263,27.6,0.0,+27.6
physician practice,0.01730,12,29,0,263,41.4,0.0,+41.4
owned nursing,0.01644,10,29,0,263,34.5,0.0,+34.5


## 5 Validate the cluster content with two representative papers

The supervisor's request — *“Die Cluster sollten zusätzlich anhand repräsentativer
Arbeiten inhaltlich validiert werden”* — is addressed below for the saved PDF
**Leiden partition at resolution 0.95**. Each cluster receives two papers drawn
from its actual membership. Cluster IDs are retained; the focus phrases are the
requested review prompts, rather than newly assigned cluster names.

Selection combines a reading of the retained abstracts and substantive PDF passages
with the existing embedding diagnostics. Each paper must address the requested
subject and substantively concern PE or sponsor-backed buyouts. Keyword occurrence
alone does not establish that relevance. The choices and content judgments are
recorded in `configs/pdf_cluster_examples.json` and checked against the exact
memberships, PDF identities, text fingerprints and quoted source blocks on every run.

**How to read the evidence:**

- **Centrality rank** orders every paper by its mean cosine similarity to the other
  papers in its cluster (rank 1 is most central). It is a diagnostic, not the selection rule.
- **Cosine silhouette** measures separation from other clusters. A negative value
  flags overlap; the PE-backed IPO example in Cluster 3 is selected for its clear
  substantive relevance despite this limitation.
- The rationale explains the paper's role and PE relevance. The scope note records
  limitations or dependencies between the two examples. Expand the evidence to read
  exact retained passages with PDF page and source-block references; extraction
  artefacts are preserved. PDF links open the local source files.

These purposively chosen examples provide qualitative support for interpreting the
requested themes. They do not establish that every member fits the theme, or that
five clusters are statistically optimal. The boundary and random examples in the
following inspection section remain useful for challenging the interpretation.

The ten selections, twenty passages, and diagnostics for all 292 papers are saved
automatically under `reports/pdf_cluster_examples/`, in a folder identified by the
review configuration's checksum. Editing the review creates a separate report.


In [38]:
from importlib import reload
from urllib.parse import quote
import batill.pdf_cluster_examples as pdf_examples
reload(pdf_examples)
from batill.storage import read_json

REVIEW_CONFIG = ROOT / 'configs/pdf_cluster_examples.json'
representative_review = read_json(REVIEW_CONFIG)
if (partition_settings.get('source') != 'saved'
        or partition_settings.get('saved') != representative_review['solution']):
    display(Markdown('These content-reviewed examples belong to **saved `leiden_k5` '
                     '(resolution 0.95)**. Select that partition in section 2 and rerun '
                     'to display them; other partitions need their own content review.'))
else:
    representatives, representative_passages, representative_diagnostics = pdf_examples.representative_papers(
        analysis, partition_settings, representative_review,
    )
    for cluster in sorted(representatives.cluster.unique()):
        focus = representative_review['focus'][str(cluster)]
        display(Markdown(f'### Cluster {cluster}\n\n**Requested focus:** {focus}'))
        for paper in representatives.loc[representatives.cluster.eq(cluster)].itertuples():
            pdf_path = ROOT / 'data/pdfs' / paper.source_filename
            if not pdf_path.is_file():
                raise FileNotFoundError(f'Missing reviewed PDF: {pdf_path}')
            pdf_url = quote('data/pdfs/' + paper.source_filename, safe='/')
            diagnostics = (f'Centrality rank {paper.centrality_rank}/{paper.cluster_papers}; '
                           f'mean within-cluster cosine {paper.mean_within_cluster_cosine:.3f}; '
                           f'cosine silhouette {paper.cosine_silhouette:+.3f}')
            if paper.cosine_silhouette < 0:
                diagnostics += ' — boundary example: overlaps other clusters'
            passages_html = []
            for passage in representative_passages.loc[representative_passages.paper_id.eq(paper.paper_id)].itertuples():
                location = escape(f'PDF page {passage.page}; block {passage.block_id}')
                passages_html.append(f'<p><a href="{pdf_url}#page={passage.page}" target="_blank">'
                                     f'{location}</a></p><blockquote style="white-space:pre-wrap">'
                                     f'{escape(passage.excerpt)}</blockquote>')
            display(HTML(
                '<div style="border-left:3px solid #46758c;padding:0 1em;margin:1em 0 1.6em">'
                f'<h4>{escape(paper.paper_id)} · {escape(paper.citation)}<br>'
                f'<a href="{pdf_url}" target="_blank">{escape(paper.title)}</a></h4>'
                f'<p><strong>Role in this pair:</strong> {escape(paper.role)}</p>'
                f'<p><strong>Why selected:</strong> {escape(paper.why_representative)}</p>'
                f'<p><strong>Private-equity relevance:</strong> {escape(paper.private_equity_focus)}</p>'
                f'<p><strong>Diagnostic context:</strong> {escape(diagnostics)}</p>'
                f'<p><strong>Scope and interpretation:</strong> {escape(paper.scope_note)}</p>'
                '<details><summary>Read the two supporting PDF passages</summary>'
                + ''.join(passages_html) + '</details></div>'
            ))
    representative_export = pdf_examples.export_representative_papers(
        ROOT, REVIEW_CONFIG, representative_review, representatives,
        representative_passages, representative_diagnostics, partition_settings,
    )
    print(f'Saved {len(representatives)} examples and {len(representative_passages)} passages:')
    print(representative_export)


### Cluster 1

**Requested focus:** employment growth

### Cluster 2

**Requested focus:** leveraged buyouts: financing and process

### Cluster 3

**Requested focus:** going private / going public

### Cluster 4

**Requested focus:** fund performance

### Cluster 5

**Requested focus:** health care and nursing homes

Saved 10 examples and 20 passages:
/Users/tilldedek/Documents/Semester 6/Bachelorarbeit/Python/batill/reports/pdf_cluster_examples/73f6535eec9cd0b7


## 8 Save evidence for your interpretation

Set `SAVE_RESULTS = True` when you want a permanent record. Each export creates a new
folder under `outputs/topic_analysis_2`; previous evidence is never replaced.

It contains full eligible word/expression rankings for both methods, cluster sizes,
membership, review papers with excerpts, excluded isolates, supporting papers for
the selected expression, and a manifest of parameters, input hashes and software versions.
Scores and proportions are saved at full precision. Plot display is not required for reproduction.

When you later name a cluster, record which expressions and papers support the name,
which papers challenge it, and the exported run folder. Inspect table noise (for example
“yes yes”), broad methodological vocabulary, synonyms and mixed subject matter before
interpreting a ranking. A descriptive label should remain open to revision.

In [44]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    exported = export_analysis(analysis, context, partition_settings, support=support,
        display_settings=dict(cluster=CLUSTER, ranking=RANKING, kind=TERM_KIND,
                              top_n=TOP_N, detail_top_n=DETAIL_TOP_N, expression=selected_expression))
    print('Saved evidence:', exported)
else:
    print('Full expression-evidence export disabled. Set SAVE_RESULTS = True to save it.')

Full expression-evidence export disabled. Set SAVE_RESULTS = True to save it.
